In [18]:
from datetime import date

import polars as pl

from airports.airportLookup import Airport_Size_Ranking, AirportLookup
from data_engineering.adsb.read_adsb import pia_or_american_ladd_icao
from data_engineering.flights.flight_type import get_flights

test_date = date(2026,6,1)
df = get_flights(test_date)
df = df.filter(pia_or_american_ladd_icao())

In [23]:
airport_lookup = AirportLookup()

def airport_size(airport_ident):
    airport = airport_lookup.get_Airport_from_airport_ident(airport_ident)
    if airport is None:
        return None
    return Airport_Size_Ranking[airport.type.name]

route_mismatches = (
    df.sort(["icao", "takeoff_time"])
    .with_columns(
        pl.col("landing_airport_ident").shift(1).over("icao").alias("prev_landing_airport_ident"),
        pl.col("landing_time").shift(1).over("icao").alias("prev_landing_time"),
        pl.col("landing_airport_ident_distance_km").shift(1).over("icao").alias("prev_landing_airport_ident_distance_km"),
        pl.col("landing_airport_elevation_difference_ft").shift(1).over("icao").alias("prev_landing_airport_elevation_difference_ft"),
    )
    .with_columns(
        pl.col("prev_landing_airport_ident")
        .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
        .alias("prev_landing_airport_in_us_faa_airspace"),
        pl.col("takeoff_airport_ident")
        .map_elements(airport_lookup.airport_ident_is_in_us_faa_airspace, return_dtype=pl.Boolean)
        .alias("takeoff_airport_in_us_faa_airspace"),
        pl.col("prev_landing_airport_ident")
        .map_elements(airport_size, return_dtype=pl.Int64)
        .alias("prev_landing_airport_size"),
        pl.col("takeoff_airport_ident")
        .map_elements(airport_size, return_dtype=pl.Int64)
        .alias("takeoff_airport_size"),
    )
    .filter(
        pl.col("prev_landing_airport_ident").is_not_null()
        & (pl.col("prev_landing_airport_ident") != pl.col("takeoff_airport_ident"))
        & pl.col("prev_landing_airport_in_us_faa_airspace")
        & pl.col("takeoff_airport_in_us_faa_airspace")
    )
    .select(
        "icao",
        "prev_landing_time",
        "prev_landing_airport_ident",
        "prev_landing_airport_size",
        "prev_landing_airport_ident_distance_km",
        "prev_landing_airport_elevation_difference_ft",
        "takeoff_time",
        "takeoff_airport_ident",
        "takeoff_airport_size",
        "takeoff_airport_ident_distance_km",
        "takeoff_airport_elevation_difference_ft",

    )
)

route_mismatches

icao,prev_landing_time,prev_landing_airport_ident,prev_landing_airport_size,prev_landing_airport_ident_distance_km,prev_landing_airport_elevation_difference_ft,takeoff_time,takeoff_airport_ident,takeoff_airport_size,takeoff_airport_ident_distance_km,takeoff_airport_elevation_difference_ft
str,datetime[ms],str,i64,f64,i64,datetime[ms],str,i64,f64,i64
"""a00323""",2026-06-01 18:09:20.140,"""14P""",1,17.541035,10573,2026-06-01 18:34:34.275,"""US-0413""",1,8.252265,9455
"""a00385""",2026-06-01 16:11:13.550,"""KEWB""",2,15.735971,670,2026-06-01 16:51:41.440,"""KTAN""",1,8.118915,1157
"""a00945""",2026-06-01 16:28:01.650,"""08IS""",1,3.756304,2625,2026-06-01 19:02:58.360,"""KMDH""",2,6.294448,5164
"""a00a5b""",2026-06-01 16:17:54.100,"""US-0996""",1,9.011479,5934,2026-06-01 19:59:44.875,"""TE62""",1,6.958539,8885
"""a010f7""",2026-06-01 16:12:27.320,"""KSUN""",2,7.309461,1057,2026-06-01 16:57:30.080,"""U93""",1,0.31634,11906
…,…,…,…,…,…,…,…,…,…,…
"""ade04b""",2026-06-01 18:47:50.730,"""5IS4""",1,3.526412,790,2026-06-01 19:27:30.840,"""US-0502""",1,3.883011,1007
"""ade04b""",2026-06-01 19:35:39.220,"""IS24""",1,11.939223,1074,2026-06-01 20:20:19,"""2T2""",1,11.878191,875
"""ade849""",2026-06-01 14:10:07.330,"""US-0639""",1,13.09674,7987,2026-06-01 20:20:36.975,"""KVTN""",2,24.007805,6304


In [ ]:
# Implement if one airport is smaller than the other. 
route_mismatches.filter(pl.col("takeoff_airport_ident_distance_km") > 100)

prev_landing_time,prev_landing_airport_ident,icao,callsign,registration,takeoff_time,takeoff_airport_ident,takeoff_airport_ident_distance_km,landing_time,landing_airport_ident,landing_airport_ident_distance_km,last_lat,last_lon,last_baro_altitude_ft,last_geom_altitude_ft,takeoff_airport_elevation_difference_ft,landing_airport_elevation_difference_ft,pia,ladd,military,interesting,aircraft_type,owner,aircraft_description,category,prev_landing_airport_in_us_faa_airspace,takeoff_airport_in_us_faa_airspace
datetime[ms],str,str,str,str,datetime[ms],str,f64,datetime[ms],str,f64,f64,f64,i64,i64,i64,i64,bool,bool,bool,bool,str,str,str,str,bool,bool
2026-06-01 05:04:44.240,"""PHOG""","""a40a18""","""""","""""",2026-06-01 22:46:36.760,"""US-0228""",295.462734,2026-06-01 23:32:58.440,"""KLAX""",1.014543,33.935051,-118.401648,null,null,41936,null,false,true,false,false,"""""","""""","""""","""""",true,true
